# Generative AI Applications

**Author:** Belal Nwiran

# Setup

In [1]:
import random
import re
import unicodedata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from collections import Counter

In [2]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


# Dataset Inspection

## Data source

The dataset is **Ashaar**, a collection of Arabic poems published on Hugging Face.

- **URL:** https://huggingface.co/datasets/arbml/ashaar
- **Downloaded:** 2026-10-03
- **Version:** commit 9b5e723df1c5b13b9e4428caff758fe2f3c737f6
- **Files:** two parquet files with 127,315 poems each (254,630 poems in total)
- **License:** released for research use only (fair use). It must not be used commercially, and poets can ask to have their work removed.
- **Origin:** the poem URLs in the data point to aldiwan.net, so the poems were collected from that website.

Only the columns needed for this project are loaded (title, meter, verses, theme, URL, poet name, poet era).

## Dataset files in this repository

The two original Ashaar files (`train-00000-of-00002.parquet` and `train-00001-of-00002.parquet`, about 280 MB in total) are **not included** in this repository because they are too large.

The repository includes only the cleaned subset:

- **File:** `abbasid_filtered.parquet`
- **Content:** 29,726 Abbasid-era poems from 634 poets, with the columns `poem title`, `poem meter`, `poem verses`, `poem theme`, `poem url`, `poet name`, `poet era`
- **Made by:** the cleaning steps in this notebook (duplicates, era filter, empty poems, odd number of half-lines, length cap)

**To rebuild the subset from the original data:**

1. Download the two parquet files from https://huggingface.co/datasets/arbml/ashaar (commit 9b5e723df1c5b13b9e4428caff758fe2f3c737f6) and place them where the notebook is located.
2. Run the notebook from the top. The cleaning cells recreate `abbasid_filtered.parquet`.

**To rerun the project without the original files:** skip the loading and cleaning cells and read the subset directly:

`poems = pd.read_parquet("abbasid_filtered.parquet")`

The subset comes from the Ashaar dataset, which is released for research use only. It must not be used commercially, and poets can ask to have their work removed. The same terms apply to the subset in this repository.

In [3]:
cols = ["poem title", "poem meter", "poem verses", "poem theme",
        "poem url", "poet name", "poet era"]
files = ["train-00000-of-00002.parquet",
         "train-00001-of-00002.parquet"]

parts = [pd.read_parquet(f, columns=cols) for f in files]
print([len(p) for p in parts])
df = pd.concat(parts, ignore_index=True)

[127315, 127315]


In [4]:
print(df.shape)
print(df.dtypes)
print(df["poem url"].duplicated().sum(), "duplicate poem URLs")

(254630, 7)
poem title        str
poem meter        str
poem verses    object
poem theme        str
poem url          str
poet name         str
poet era          str
dtype: object
38 duplicate poem URLs


In [5]:
print(df["poet era"].value_counts())

poet era
العصر الحديث        55137
العصر العباسي       30413
العصر المملوكي      19059
العصر العثماني      11872
المغرب والأندلس      6614
العصر الفاطمي        4939
العصر الأندلسي       4770
العصر الأموي         3958
العصر الأيوبي        3125
المخضرمين            1975
العصر الجاهلي        1873
قبل الإسلام          1786
عصر بين الدولتين     1416
العصر الإسلامي        484
Name: count, dtype: int64


The dataset covers 14 eras. The Abbasid era has 30,413 poems, the largest after the modern era.

## Data structure

Each row is one poem. The `poem verses` column is a list of **half-lines (hemistichs)**. A line of poetry (*bayt*) has two halves: the first half (*ṣadr*) and the second half (*ʿajuz*). The list alternates between them, so the number of lines is half the number of items.

The cell below shows the first five lines of one poem, with the two halves side by side.

# Dataset Cleaning

In [6]:
def add_length_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Add hemistich count and character count per poem."""
    df = df.copy()
    df["n_hemistichs"] = df["poem verses"].apply(len)
    df["n_chars"] = df["poem verses"].apply(lambda v: sum(len(h) for h in v))
    print(f"add_length_columns: added n_hemistichs and n_chars for {len(df):,} poems")
    return df

In [7]:
def filter_era(df: pd.DataFrame, era: str) -> pd.DataFrame:
    """Keep only poems whose poet belongs to the given era."""
    df = df.copy()
    before = len(df)
    df = df[df["poet era"] == era]
    print(f"filter_era: kept {len(df):,} of {before:,} poems (era = {era})")
    return df

In [8]:
def drop_empty_poems(df: pd.DataFrame) -> pd.DataFrame:
    """Drop poems with no hemistichs."""
    df = df.copy()
    before = len(df)
    df = df[df["n_hemistichs"] > 0]
    print(f"drop_empty_poems: dropped {before - len(df):,} poems")
    return df

In [9]:
def drop_odd_hemistichs(df: pd.DataFrame) -> pd.DataFrame:
    """Drop poems with an odd number of hemistichs (broken sadr/ajuz pairing)."""
    df = df.copy()
    before = len(df)
    df = df[df["n_hemistichs"] % 2 == 0]
    print(f"drop_odd_hemistichs: dropped {before - len(df):,} poems")
    return df

In [10]:
def drop_long_poems(df: pd.DataFrame, max_hemistichs: int) -> pd.DataFrame:
    """Drop poems longer than max_hemistichs (long urjuzas, likely merged scrapes)."""
    df = df.copy()
    before = len(df)
    chars_before = df["n_chars"].sum()
    df = df[df["n_hemistichs"] <= max_hemistichs]
    share = 1 - df["n_chars"].sum() / chars_before
    print(
        f"drop_long_poems: dropped {before - len(df):,} poems "
        f"(> {max_hemistichs} hemistichs), {share:.1%} of characters"
    )
    return df

In [11]:
def drop_duplicate_poems(df: pd.DataFrame) -> pd.DataFrame:
    """Drop poems that appear twice (same poem URL), keeping the first."""
    df = df.copy()
    before = len(df)
    df = df.drop_duplicates(subset="poem url", keep="first")
    print(f"drop_duplicate_poems: dropped {before - len(df):,} poems")
    return df

In [12]:
ERA = "العصر العباسي"
CAP_QUANTILE = 0.99

poems = add_length_columns(df)
poems = filter_era(poems, ERA)
poems = drop_duplicate_poems(poems)
poems = drop_empty_poems(poems)
poems = drop_odd_hemistichs(poems)

max_hemistichs = int(poems["n_hemistichs"].quantile(CAP_QUANTILE))
print(f"Length cap ({CAP_QUANTILE:.0%} quantile): {max_hemistichs} hemistichs")
poems = drop_long_poems(poems, max_hemistichs)

poems = poems.reset_index(drop=True)
print(f"{len(poems):,} poems, {poems['n_chars'].sum() / 1e6:.1f}M characters")

add_length_columns: added n_hemistichs and n_chars for 254,630 poems
filter_era: kept 30,413 of 254,630 poems (era = العصر العباسي)
drop_duplicate_poems: dropped 0 poems
drop_empty_poems: dropped 1 poems
drop_odd_hemistichs: dropped 390 poems
Length cap (99% quantile): 156 hemistichs
drop_long_poems: dropped 296 poems (> 156 hemistichs), 12.6% of characters
29,726 poems, 15.6M characters


## Cleaning rules

The cleaning has five steps. Each step prints how many poems it removes.

1. **Remove duplicate poems.** 38 poem URLs appear twice in the full dataset, but none of the duplicate pairs are in the Abbasid era, so this step removes 0 poems. It stays as a safeguard.
2. **Keep the Abbasid era only.** The dataset is large, so one era is used. This gives one consistent style and enough text. The Abbasid era is the largest era after the modern era (modern poems may still be under copyright).
3. **Remove empty poems.** A poem with no text cannot be used.
4. **Remove poems with an odd number of half-lines.** A half-line is missing or extra, so the pairing of first and second halves is broken for the rest of the poem.
5. **Remove very long poems.** The limit is the 99th percentile of the number of half-lines (156 in the first run). About 1% of poems are removed, but they hold 12.6% of all characters, so a few very long poems would otherwise have a large influence on what the model learns.

In [13]:
poet_share = poems.groupby("poet name")["n_chars"].sum().sort_values(ascending=False)
print(poems["poet name"].nunique(), "poets")
print((poet_share.head(10) / poet_share.sum()).round(3))

634 poets
poet name
البحتري              0.068
ابن الرومي           0.067
الشريف الرضي         0.063
أبو العلاء المعري    0.051
مهيار الديلمي        0.043
ابن نباتة السعدي     0.034
ابو نواس             0.033
أبو تمام             0.032
السري الرفاء         0.028
الصنوبري             0.028
Name: n_chars, dtype: float64


The data is concentrated in a few poets: the top 10 poets write 44.7% of the text, and the largest, البحتري, writes 6.8%. The model will therefore mostly learn the style of these poets. This is discussed in the ethics section of the report.

In [14]:
poems.to_parquet("abbasid_filtered.parquet", index=False)

After cleaning, 29,726 poems remain, with 15.6 million characters from 634 poets. The cleaned data is saved to `abbasid_filtered.parquet`.

In [15]:
s = poems.iloc[0]
print(s["poet name"], "|", s["poem title"], "|", s["poem meter"])
for sadr, ajuz in zip(s["poem verses"][0:10:2], s["poem verses"][1:10:2]):
    print(sadr, "   ...   ", ajuz)

الشريف الرضي | ودجى هتكت قناعه | بحر مجزوء الكامل
وَدُجىً هَتَكتُ قِناعَهُ    ...    عَن وَجهِ طامِسَةٍ خَفِيَّه
تَسري كَواكِبُهُ إِلى ال    ...    إِصباحِ وَاللَيلُ المَطِيَّه
وَالنَجمُ وَجهُ مُقَبَّلٍ    ...    وَالبَدرُ مِرآةٌ صَدِيَّه


In [16]:
text = "".join("".join(v) for v in poems["poem verses"])
counts = Counter(text)
print(f"{len(text):,} characters, {len(counts)} unique")

def is_letter(c: str) -> bool:
    # Arabic letters, excluding tatweel (U+0640)
    return "\u0621" <= c <= "\u063A" or "\u0641" <= c <= "\u064A"

def is_diacritic(c: str) -> bool:
    return "\u064B" <= c <= "\u0652" or c == "\u0670"

print(sum(is_letter(c) for c in counts), "Arabic letters")
print(sum(is_diacritic(c) for c in counts), "diacritic marks")

for c, n in counts.most_common():
    if not is_letter(c) and not is_diacritic(c):
        print(repr(c), f"U+{ord(c):04X}", unicodedata.name(c, "?"), f"{n:,}")

15,612,370 characters, 74 unique
36 Arabic letters
8 diacritic marks
' ' U+0020 SPACE 1,790,601
'ـ' U+0640 ARABIC TATWEEL 1,141,081
'،' U+060C ARABIC COMMA 602
'.' U+002E FULL STOP 427
'"' U+0022 QUOTATION MARK 373
':' U+003A COLON 189
'؟' U+061F ARABIC QUESTION MARK 167
'!' U+0021 EXCLAMATION MARK 38
'؛' U+061B ARABIC SEMICOLON 15
'(' U+0028 LEFT PARENTHESIS 14
')' U+0029 RIGHT PARENTHESIS 14
'-' U+002D HYPHEN-MINUS 13
'@' U+0040 COMMERCIAL AT 8
'\u200d' U+200D ZERO WIDTH JOINER 6
'E' U+0045 LATIN CAPITAL LETTER E 4
'[' U+005B LEFT SQUARE BRACKET 3
'C' U+0043 LATIN CAPITAL LETTER C 2
'N' U+004E LATIN CAPITAL LETTER N 2
'T' U+0054 LATIN CAPITAL LETTER T 2
'R' U+0052 LATIN CAPITAL LETTER R 2
']' U+005D RIGHT SQUARE BRACKET 2
'4' U+0034 DIGIT FOUR 2
'ک' U+06A9 ARABIC LETTER KEHEH 1
'/' U+002F SOLIDUS 1
'=' U+003D EQUALS SIGN 1
'8' U+0038 DIGIT EIGHT 1
'a' U+0061 LATIN SMALL LETTER A 1
'7' U+0037 DIGIT SEVEN 1
'\u200c' U+200C ZERO WIDTH NON-JOINER 1
'd' U+0064 LATIN SMALL LETTER D 1


## Text normalization

The model reads the poems one character at a time, so the text is cleaned to keep only the Arabic letters and the space. The rules come from the character count of the cleaned subset above (74 unique characters, 15.6 million in total):

1. **Remove diacritics (tashkeel).** They are used inconsistently. In the sample poem above, the same word appears with different marks (for example رَبَنا and رَبّنا), so the model would learn noise.
2. **Remove tatweel (ـ).** It only stretches letters visually and has no meaning. It is 1,141,081 characters, 7.3% of the text.
3. **Remove zero-width joiner and non-joiner.** There are only 7 of them, and they sit inside words.
4. **Replace every other character with a space.** This covers punctuation, digits, Latin letters, brackets and symbols. Together they are about 1,900 characters (0.01%), too rare for the model to learn.
5. **Collapse repeated spaces** and trim the ends of each half-line.
6. **Remove poems with an empty half-line** after cleaning ([n] poems), because the pairing of first and second halves would break.

Letter variants are not merged (for example أ, إ and ا stay different), so the spelling of the poems stays as written.

After normalization the text has [X] million characters and [U] unique characters (the Arabic letters and the space).

In [17]:
DIACRITICS = re.compile("[\u064B-\u0652\u0670]")
ZERO_WIDTH = re.compile("[\u200c\u200d]")
NOT_LETTER_OR_SPACE = re.compile("[^\u0621-\u063A\u0641-\u064A ]")
MANY_SPACES = re.compile(" +")


def normalize_text(text: str) -> str:
    """Keep only Arabic letters and single spaces."""
    text = DIACRITICS.sub("", text)
    text = text.replace("\u0640", "")          # tatweel
    text = ZERO_WIDTH.sub("", text)
    text = NOT_LETTER_OR_SPACE.sub(" ", text)  # punctuation, digits, Latin, etc.
    text = MANY_SPACES.sub(" ", text)
    return text.strip()


def normalize_poems(df: pd.DataFrame) -> pd.DataFrame:
    """Normalize every half-line of every poem and update n_chars."""
    df = df.copy()
    chars_before = df["n_chars"].sum()
    df["poem verses"] = df["poem verses"].apply(lambda v: [normalize_text(h) for h in v])
    df["n_chars"] = df["poem verses"].apply(lambda v: sum(len(h) for h in v))
    chars_after = df["n_chars"].sum()
    print(f"normalize_poems: {chars_before:,} -> {chars_after:,} characters "
          f"({1 - chars_after / chars_before:.1%} removed)")
    return df


def drop_poems_with_empty_halves(df: pd.DataFrame) -> pd.DataFrame:
    """Drop poems where a half-line became empty (breaks the pairing)."""
    df = df.copy()
    before = len(df)
    has_empty = df["poem verses"].apply(lambda v: any(h == "" for h in v))
    df = df[~has_empty]
    print(f"drop_poems_with_empty_halves: dropped {before - len(df):,} poems")
    return df

In [18]:
example = poems.iloc[0]["poem verses"][0]
print("before:", example)
print("after: ", normalize_text(example))

poems = normalize_poems(poems)
poems = drop_poems_with_empty_halves(poems)
poems = poems.reset_index(drop=True)

before: وَدُجىً هَتَكتُ قِناعَهُ
after:  ودجى هتكت قناعه
normalize_poems: 15,612,370 -> 11,014,129 characters (29.5% removed)
drop_poems_with_empty_halves: dropped 19 poems


In [19]:
text = "".join("".join(v) for v in poems["poem verses"])
print(f"{len(poems):,} poems, {len(text):,} characters, {len(set(text))} unique")
print("".join(sorted(set(text))))

29,707 poems, 11,011,086 characters, 37 unique
 ءآأؤإئابةتثجحخدذرزسشصضطظعغفقكلمنهوىي


In [20]:
HALF_SEP = "|"
LINE_END = "\n"
POEM_END = "#"


def poem_to_text(verses: list[str]) -> str:
    """Join half-lines into lines, and add the end-of-poem marker."""
    assert len(verses) % 2 == 0, "odd number of half-lines"
    lines = [f"{sadr}{HALF_SEP}{ajuz}" for sadr, ajuz in zip(verses[0::2], verses[1::2])]
    return LINE_END.join(lines) + LINE_END + POEM_END + LINE_END


def add_text_column(df: pd.DataFrame) -> pd.DataFrame:
    """Add one training text per poem, in the format used by the model."""
    df = df.copy()
    df["text"] = df["poem verses"].apply(poem_to_text)
    print(f"add_text_column: built text for {len(df):,} poems, "
          f"{df['text'].str.len().sum():,} characters")
    return df

In [21]:
poems = add_text_column(poems)

print(poems["text"].iloc[0][:300])

corpus = "".join(poems["text"])
print(f"{len(set(corpus))} unique characters")
print(repr("".join(sorted(set(corpus) - set(text)))))
print(poems["text"].str.len().describe())

add_text_column: built text for 29,707 poems, 11,557,072 characters
ودجى هتكت قناعه|عن وجه طامسة خفيه
تسري كواكبه إلى ال|إصباح والليل المطيه
والنجم وجه مقبل|والبدر مرآة صديه
#

40 unique characters
'\n#|'
count    29707.000000
mean       389.035312
std        578.820012
min         27.000000
25%        105.000000
50%        176.000000
75%        358.000000
max       4287.000000
Name: text, dtype: float64


## Text format for the model

The model needs to know where each half-line, each line and each poem ends. Three symbols that do not appear in the cleaned text are added:

- `|` between the first half (*ṣadr*) and the second half (*ʿajuz*) of a line
- a new line at the end of each line
- `#` after the last line of a poem, so the model learns where a poem stops

Each poem becomes one text string. Some words are split across the two halves of a line, so the separator is not always placed between two words.

The vocabulary is therefore 40 symbols: the 36 Arabic letters, the space and these three symbols.